## FE

In [12]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import accuracy_score, classification_report, brier_score_loss
from os.path import join

In [13]:
def lacz_dataframes(df_sezon, df_turniej):
    """
    Łączy dataframe z sezonu sportowego i dataframe z turnieju, 
    dodając kolumnę 'isTourney' oznaczającą źródło danych.
    
    Parameters:
    df_sezon (pandas.DataFrame): Dataframe zawierający dane z sezonu sportowego
    df_turniej (pandas.DataFrame): Dataframe zawierający dane z turnieju
    
    Returns:
    pandas.DataFrame: Połączony dataframe z dodaną kolumną 'isTourney'
    """
    # Dodanie kolumny 'isTourney' z wartościami
    df_sezon['isTourney'] = 0
    df_turniej['isTourney'] = 1
    
    # Połączenie dataframe'ów
    df_polaczony = pd.concat([df_sezon, df_turniej], ignore_index=True)
    
    return df_polaczony

In [14]:
data_path = 'data'

In [15]:
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))

In [16]:
def przygotuj_dane_do_prognozy(df_sezon, df_turniej, year=2024):
    """
    Łączy dane z sezonów i turniejów, tworząc zbiór treningowy i testowy
    do prognozowania wyników turnieju w określonym roku.
    
    Parameters:
    df_sezon (pandas.DataFrame): DataFrame z danymi z sezonów
    df_turniej (pandas.DataFrame): DataFrame z danymi z turniejów
    year (int): Rok docelowy do prognozowania (domyślnie 2024)
    
    Returns:
    tuple: (X_train, X_test, y_train, y_test, model, cechy_waznosci, df_train, df_test, scaler)
    """
    # Dodanie kolumny 'isTourney'
    df_sezon['isTourney'] = 0
    df_turniej['isTourney'] = 1
    
    # Połączenie danych
    df_polaczony = pd.concat([df_sezon, df_turniej], ignore_index=True)
    
    # Podział na dane treningowe i testowe
    # Dane treningowe: sezony bieżącego roku i wszystkie wcześniejsze turnieje i sezony
    df_train = df_polaczony[
        ((df_polaczony['Season'] == year) & (df_polaczony['isTourney'] == 0)) | 
        (df_polaczony['Season'] < year)
    ]
    
    # Dane testowe: turniej bieżącego roku (którego wyniki chcemy prognozować)
    df_test = df_polaczony[
        (df_polaczony['Season'] == year) & (df_polaczony['isTourney'] == 1)
    ]
    
    # Tworzenie nowych cech (statystyki różnicowe i stosunkowe)
    cechy_roznicowe = []
    cechy_stosunkowe = []
    
    for col in ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']:
        # Różnica między zwycięzcą a przegranym
        df_train[f'Diff_{col}'] = df_train[f'W{col}'] - df_train[f'L{col}']
        df_test[f'Diff_{col}'] = df_test[f'W{col}'] - df_test[f'L{col}']
        cechy_roznicowe.append(f'Diff_{col}')
        
        # Stosunek zwycięzcy do przegranego (z obsługą dzielenia przez zero)
        df_train[f'Ratio_{col}'] = df_train[f'W{col}'] / df_train[f'L{col}'].replace(0, 0.001)
        df_test[f'Ratio_{col}'] = df_test[f'W{col}'] / df_test[f'L{col}'].replace(0, 0.001)
        cechy_stosunkowe.append(f'Ratio_{col}')
    
    # Utworzenie dodatkowych statystyk meczowych
    # Skuteczność rzutów z gry
    df_train['W_FG_Pct'] = df_train['WFGM'] / df_train['WFGA'].replace(0, 0.001) * 100
    df_train['L_FG_Pct'] = df_train['LFGM'] / df_train['LFGA'].replace(0, 0.001) * 100
    df_test['W_FG_Pct'] = df_test['WFGM'] / df_test['WFGA'].replace(0, 0.001) * 100
    df_test['L_FG_Pct'] = df_test['LFGM'] / df_test['LFGA'].replace(0, 0.001) * 100
    
    # Skuteczność rzutów za 3 punkty
    df_train['W_3P_Pct'] = df_train['WFGM3'] / df_train['WFGA3'].replace(0, 0.001) * 100
    df_train['L_3P_Pct'] = df_train['LFGM3'] / df_train['LFGA3'].replace(0, 0.001) * 100
    df_test['W_3P_Pct'] = df_test['WFGM3'] / df_test['WFGA3'].replace(0, 0.001) * 100
    df_test['L_3P_Pct'] = df_test['LFGM3'] / df_test['LFGA3'].replace(0, 0.001) * 100
    
    # Skuteczność rzutów wolnych
    df_train['W_FT_Pct'] = df_train['WFTM'] / df_train['WFTA'].replace(0, 0.001) * 100
    df_train['L_FT_Pct'] = df_train['LFTM'] / df_train['LFTA'].replace(0, 0.001) * 100
    df_test['W_FT_Pct'] = df_test['WFTM'] / df_test['WFTA'].replace(0, 0.001) * 100
    df_test['L_FT_Pct'] = df_test['LFTM'] / df_test['LFTA'].replace(0, 0.001) * 100
    
    # Suma zbiórek
    df_train['W_Total_Reb'] = df_train['WOR'] + df_train['WDR']
    df_train['L_Total_Reb'] = df_train['LOR'] + df_train['LDR']
    df_test['W_Total_Reb'] = df_test['WOR'] + df_test['WDR']
    df_test['L_Total_Reb'] = df_test['LOR'] + df_test['LDR']
    
    # Posiadanie piłki (Assist to Turnover ratio)
    df_train['W_Ast_TO_Ratio'] = df_train['WAst'] / df_train['WTO'].replace(0, 0.001)
    df_train['L_Ast_TO_Ratio'] = df_train['LAst'] / df_train['LTO'].replace(0, 0.001)
    df_test['W_Ast_TO_Ratio'] = df_test['WAst'] / df_test['WTO'].replace(0, 0.001)
    df_test['L_Ast_TO_Ratio'] = df_test['LAst'] / df_test['LTO'].replace(0, 0.001)
    
    # Różnice w procentach i innych statystykach
    dodatkowe_cechy = [
        'W_FG_Pct', 'L_FG_Pct', 'W_3P_Pct', 'L_3P_Pct', 'W_FT_Pct', 'L_FT_Pct',
        'W_Total_Reb', 'L_Total_Reb', 'W_Ast_TO_Ratio', 'L_Ast_TO_Ratio',
        'isTourney', 'NumOT'
    ]
    
    # Wybór cech do modelu
    cechy = cechy_roznicowe + cechy_stosunkowe + dodatkowe_cechy
    
    # Przykładowe cechy klasyfikacyjne - w rzeczywistym modelu potrzebne byłyby 
    # specjalnie przygotowane dane dla klasyfikacji drużyn
    # Tutaj używamy prostej klasyfikacji basedu on różnicy punktów
    df_train['WinMargin'] = df_train['WScore'] - df_train['LScore']
    df_test['WinMargin'] = df_test['WScore'] - df_test['LScore']
    
    # Tworzenie zmiennej celu (czy margin wygranej > 10 punktów)
    df_train['BigWin'] = (df_train['WinMargin'] > 10).astype(int)
    df_test['BigWin'] = (df_test['WinMargin'] > 10).astype(int)
    
    # Przygotowanie danych treningowych i testowych
    X_train = df_train[cechy]
    y_train = df_train['BigWin']
    X_test = df_test[cechy]
    y_test = df_test['BigWin']
    
    # Standaryzacja cech
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Trenowanie modelu 
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    # Obliczanie ważności cech
    cechy_waznosci = pd.DataFrame({
        'Cecha': cechy,
        'Ważność': model.feature_importances_
    }).sort_values('Ważność', ascending=False)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, model, cechy_waznosci, df_train, df_test, scaler


In [17]:
def prognozuj_turniej(df_sezon, df_turniej, year=2024):
    """
    Prognozuje wyniki turnieju na podstawie danych historycznych.
    Dodana funkcjonalność Brier Score do oceny jakości predykcji.
    
    Parameters:
    df_sezon (pandas.DataFrame): DataFrame z danymi z sezonów
    df_turniej (pandas.DataFrame): DataFrame z danymi z turniejów
    year (int): Rok docelowy do prognozowania (domyślnie 2024)
    
    Returns:
    dict: Słownik z wynikami prognozy, statystykami i modelem
    """
    # Przygotowanie danych
    X_train, X_test, y_train, y_test, model, cechy_waznosci, df_train, df_test, scaler = przygotuj_dane_do_prognozy(
        df_sezon, df_turniej, year
    )
    
    # Prognozowanie
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    # Ocena modelu
    accuracy = accuracy_score(y_test, y_pred)
    raport = classification_report(y_test, y_pred, output_dict=True)
    
    # Obliczenie Brier Score
    # Dla modelu binarnego, bierzemy prawdopodobieństwo pozytywnej klasy (klasy 1)
    brier_score = brier_score_loss(y_test, y_pred_proba[:, 1])
    
    # Statystyki dotyczące turnieju
    statystyki_turnieju = {
        'liczba_meczy': len(df_test),
        'srednia_roznica_punktow': df_test['WinMargin'].mean(),
        'max_roznica_punktow': df_test['WinMargin'].max(),
        'procent_duzych_wygranych': df_test['BigWin'].mean() * 100,
        'srednia_skutecznosc_zwyciezcow': df_test['W_FG_Pct'].mean(),
        'srednia_skutecznosc_przegranych': df_test['L_FG_Pct'].mean(),
        'srednia_roznica_asyst': (df_test['WAst'] - df_test['LAst']).mean(),
        'srednia_roznica_zbiorek': (df_test['W_Total_Reb'] - df_test['L_Total_Reb']).mean()
    }

    # Statystyki porównawcze między sezonem a turniejem
    # Tylko dla danych z docelowego roku
    sezon_rok = df_train[(df_train['Season'] == year) & (df_train['isTourney'] == 0)]
    
    porownanie = {
        'sezon_srednia_roznica': sezon_rok['WinMargin'].mean() if not sezon_rok.empty else 0,
        'turniej_srednia_roznica': df_test['WinMargin'].mean() if not df_test.empty else 0,
        'sezon_procent_duzych_wygranych': sezon_rok['BigWin'].mean() * 100 if not sezon_rok.empty else 0,
        'turniej_procent_duzych_wygranych': df_test['BigWin'].mean() * 100 if not df_test.empty else 0,
        'sezon_srednia_skutecznosc_W': sezon_rok['W_FG_Pct'].mean() if not sezon_rok.empty else 0,
        'turniej_srednia_skutecznosc_W': df_test['W_FG_Pct'].mean() if not df_test.empty else 0
    }

    # Wyliczenie czynników najlepiej przewidujących wygraną
    top_czynniki = cechy_waznosci.head(10).to_dict('records')
    
    wyniki = {
        'rok': year,
        'dokładność_modelu': accuracy,
        'brier_score': brier_score,  # Dodany Brier Score
        'raport_klasyfikacji': raport,
        'statystyki_turnieju': statystyki_turnieju,
        'porównanie_sezon_turniej': porownanie,
        'najważniejsze_czynniki': top_czynniki,
        'model': model,
        'scaler': scaler,  # Zachowujemy skaler do późniejszych predykcji
        'cechy': cechy_waznosci['Cecha'].tolist()  # Lista cech do późniejszego użycia
    }
    
    return wyniki

In [18]:
def generuj_ciekawe_statystyki(df_sezon, df_turniej):
    """
    Generuje ciekawe statystyki i porównania między sezonami a turniejami.
    
    Parameters:
    df_sezon (pandas.DataFrame): DataFrame z danymi z sezonów
    df_turniej (pandas.DataFrame): DataFrame z danymi z turniejów
    
    Returns:
    dict: Słownik z ciekawymi statystykami
    """
    # Dodanie kolumny isTourney
    df_sezon['isTourney'] = 0
    df_turniej['isTourney'] = 1
    
    # Łączenie danych
    df = pd.concat([df_sezon, df_turniej], ignore_index=True)
    
    # Dodanie kilku użytecznych kolumn
    df['WinMargin'] = df['WScore'] - df['LScore']
    df['W_FG_Pct'] = df['WFGM'] / df['WFGA'].replace(0, 0.001) * 100
    df['L_FG_Pct'] = df['LFGM'] / df['LFGA'].replace(0, 0.001) * 100
    df['W_3P_Pct'] = df['WFGM3'] / df['WFGA3'].replace(0, 0.001) * 100
    df['L_3P_Pct'] = df['LFGM3'] / df['LFGA3'].replace(0, 0.001) * 100
    df['W_FT_Pct'] = df['WFTM'] / df['WFTA'].replace(0, 0.001) * 100
    df['L_FT_Pct'] = df['LFTM'] / df['LFTA'].replace(0, 0.001) * 100
    df['W_Total_Reb'] = df['WOR'] + df['WDR']
    df['L_Total_Reb'] = df['LOR'] + df['LDR']
    df['Total_Points'] = df['WScore'] + df['LScore']
    
    # 1. Porównanie sezon vs turniej
    sezon = df[df['isTourney'] == 0]
    turniej = df[df['isTourney'] == 1]
    
    porownanie_ogolne = {
        'średnia_różnica_punktów_sezon': sezon['WinMargin'].mean(),
        'średnia_różnica_punktów_turniej': turniej['WinMargin'].mean(),
        'średnia_suma_punktów_sezon': sezon['Total_Points'].mean(),
        'średnia_suma_punktów_turniej': turniej['Total_Points'].mean(),
        'skuteczność_rzutów_zwycięzców_sezon': sezon['W_FG_Pct'].mean(),
        'skuteczność_rzutów_zwycięzców_turniej': turniej['W_FG_Pct'].mean(),
        'skuteczność_rzutów_przegranych_sezon': sezon['L_FG_Pct'].mean(),
        'skuteczność_rzutów_przegranych_turniej': turniej['L_FG_Pct'].mean()
    }
    
    # 2. Statystyki dla każdego roku
    statystyki_roczne = []
    for year in sorted(df['Season'].unique()):
        rok_sezon = sezon[sezon['Season'] == year]
        rok_turniej = turniej[turniej['Season'] == year]
        
        if not rok_sezon.empty and not rok_turniej.empty:
            stats_rok = {
                'rok': year,
                'liczba_meczy_sezon': len(rok_sezon),
                'liczba_meczy_turniej': len(rok_turniej),
                'średnia_różnica_punktów_sezon': rok_sezon['WinMargin'].mean(),
                'średnia_różnica_punktów_turniej': rok_turniej['WinMargin'].mean(),
                'procent_meczy_z_dogrywką_sezon': (rok_sezon['NumOT'] > 0).mean() * 100,
                'procent_meczy_z_dogrywką_turniej': (rok_turniej['NumOT'] > 0).mean() * 100
            }
            statystyki_roczne.append(stats_rok)
    
    # 3. Czynniki najbardziej różniące mecze turniejowe od sezonowych
    roznice = {}
    for col in ['WinMargin', 'W_FG_Pct', 'L_FG_Pct', 'W_3P_Pct', 'L_3P_Pct', 
                'W_FT_Pct', 'L_FT_Pct', 'W_Total_Reb', 'L_Total_Reb', 
                'WAst', 'LAst', 'WTO', 'LTO', 'WStl', 'LStl', 'WBlk', 'LBlk']:
        if col in df.columns:
            roznica = turniej[col].mean() - sezon[col].mean()
            roznice[col] = roznica
    
    # 4. Analiza przewagi "home court" vs mecze na neutralnym terenie
    wygrane_u_siebie = df[df['WLoc'] == 'H']
    wygrane_na_wyjeździe = df[df['WLoc'] == 'A']
    wygrane_neutralne = df[df['WLoc'] == 'N']
    
    przewaga_lokalizacji = {
        'procent_meczy_gospodarzy': len(wygrane_u_siebie) / len(df) * 100,
        'procent_meczy_gości': len(wygrane_na_wyjeździe) / len(df) * 100,
        'procent_meczy_neutralnych': len(wygrane_neutralne) / len(df) * 100,
        'średnia_różnica_punktów_gospodarze': wygrane_u_siebie['WinMargin'].mean(),
        'średnia_różnica_punktów_goście': wygrane_na_wyjeździe['WinMargin'].mean(),
        'średnia_różnica_punktów_neutralne': wygrane_neutralne['WinMargin'].mean()
    }
    
    # 5. Analiza znaczenia różnych statystyk dla zwycięstwa
    korelacje_z_wygrana = {}
    for col in ['W_FG_Pct', 'W_3P_Pct', 'W_FT_Pct', 'W_Total_Reb', 'WAst', 'WTO', 'WStl', 'WBlk']:
        if col in df.columns and f'L{col[1:]}' in df.columns:
            roznica = df[col] - df[f'L{col[1:]}']
            korelacja = np.corrcoef(roznica, df['WinMargin'])[0, 1]
            korelacje_z_wygrana[f'korelacja_{col}_z_wygrana'] = korelacja
    
    # 6. Czy turnieje są bardziej przewidywalne niż sezony?
    # Sprawdź, czy różnica w statystykach lepiej koreluje z wynikiem w turnieju niż w sezonie
    przewidywalnosc = {}
    for stat in ['FG_Pct', '3P_Pct', 'FT_Pct', 'Total_Reb', 'Ast', 'TO', 'Stl', 'Blk']:
        if f'W_{stat}' in sezon.columns and f'L_{stat}' in sezon.columns:
            sezon_diff = sezon[f'W_{stat}'] - sezon[f'L_{stat}']
            turniej_diff = turniej[f'W_{stat}'] - turniej[f'L_{stat}']
            
            sezon_corr = np.corrcoef(sezon_diff, sezon['WinMargin'])[0, 1]
            turniej_corr = np.corrcoef(turniej_diff, turniej['WinMargin'])[0, 1]
            
            przewidywalnosc[f'korelacja_{stat}_sezon'] = sezon_corr
            przewidywalnosc[f'korelacja_{stat}_turniej'] = turniej_corr
            przewidywalnosc[f'różnica_korelacji_{stat}'] = turniej_corr - sezon_corr
    
    # 7. Wyliczenie "upset rate" - jak często niżej notowane drużyny wygrywają w turnieju vs sezon
    # W rzeczywistym modelu wymagałoby to dodatkowych danych o rankingach drużyn
    
    # Wynik
    wyniki = {
        'porównanie_ogólne': porownanie_ogolne,
        'statystyki_roczne': statystyki_roczne,
        'główne_różnice_sezon_turniej': sorted(roznice.items(), key=lambda x: abs(x[1]), reverse=True),
        'analiza_przewagi_lokalizacji': przewaga_lokalizacji,
        'korelacje_statystyk_z_wygraną': korelacje_z_wygrana,
        'przewidywalność_sezon_vs_turniej': przewidywalnosc
    }
    
    return wyniki

In [20]:
df_polaczony = lacz_dataframes(MRegularSeasonDetailedResults, MNCAATourneyDetailedResults)

In [21]:
przygotuj_dane_do_prognozy(MRegularSeasonDetailedResults, MNCAATourneyDetailedResults, year=2024)

C:\Users\micha\AppData\Local\Temp\ipykernel_31620\1030071239.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train[f'Diff_{col}'] = df_train[f'W{col}'] - df_train[f'L{col}']
C:\Users\micha\AppData\Local\Temp\ipykernel_31620\1030071239.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test[f'Diff_{col}'] = df_test[f'W{col}'] - df_test[f'L{col}']
C:\Users\micha\AppData\Local\Temp\ipykernel_31620\1030071239.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Da

(array([[ 2.84725859e-01,  6.86866095e-01, -3.12441623e-02, ...,
         -1.32443090e-02, -1.07760855e-01, -2.25372832e-01],
        [-3.16255394e-01, -4.44770732e-01,  2.25476986e-01, ...,
         -1.05871367e-02, -1.07760855e-01, -2.25372832e-01],
        [-3.16255394e-01, -1.57640756e+00,  9.95640431e-01, ...,
         -7.39852993e-03, -1.07760855e-01, -2.25372832e-01],
        ...,
        [ 8.85707112e-01, -4.44770732e-01,  2.25476986e-01, ...,
         -4.89881943e-04,  9.27980759e+00, -2.25372832e-01],
        [-3.16255394e-01,  6.86866095e-01, -2.87965311e-01, ...,
         -8.99283331e-03,  9.27980759e+00, -2.25372832e-01],
        [ 8.43987750e-02, -5.57934415e-01, -2.87965311e-01, ...,
         -1.05871367e-02,  9.27980759e+00, -2.25372832e-01]]),
 array([[ 1.68701545e+00, -8.97425463e-01,  2.25476986e-01, ...,
          1.65160208e-02,  9.27980759e+00, -2.25372832e-01],
        [ 6.85380028e-01, -1.05279684e-01, -3.12441623e-02, ...,
          1.37829293e-02,  9.27980759e

In [22]:
prognozuj_turniej(MRegularSeasonDetailedResults, MNCAATourneyDetailedResults, year=2024)

C:\Users\micha\AppData\Local\Temp\ipykernel_31620\1030071239.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train[f'Diff_{col}'] = df_train[f'W{col}'] - df_train[f'L{col}']
C:\Users\micha\AppData\Local\Temp\ipykernel_31620\1030071239.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test[f'Diff_{col}'] = df_test[f'W{col}'] - df_test[f'L{col}']
C:\Users\micha\AppData\Local\Temp\ipykernel_31620\1030071239.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Da

{'rok': 2024,
 'dokładność_modelu': 0.9701492537313433,
 'brier_score': 0.0356223880597015,
 'raport_klasyfikacji': {'0': {'precision': 0.9285714285714286,
   'recall': 1.0,
   'f1-score': 0.962962962962963,
   'support': 26.0},
  '1': {'precision': 1.0,
   'recall': 0.9512195121951219,
   'f1-score': 0.975,
   'support': 41.0},
  'accuracy': 0.9701492537313433,
  'macro avg': {'precision': 0.9642857142857143,
   'recall': 0.975609756097561,
   'f1-score': 0.9689814814814814,
   'support': 67.0},
  'weighted avg': {'precision': 0.9722814498933902,
   'recall': 0.9701492537313433,
   'f1-score': 0.9703289110005527,
   'support': 67.0}},
 'statystyki_turnieju': {'liczba_meczy': 67,
  'srednia_roznica_punktow': 14.388059701492537,
  'max_roznica_punktow': 40,
  'procent_duzych_wygranych': 61.19402985074627,
  'srednia_skutecznosc_zwyciezcow': 49.06981205541794,
  'srednia_skutecznosc_przegranych': 38.99637010078203,
  'srednia_roznica_asyst': 4.880597014925373,
  'srednia_roznica_zbiorek'

In [23]:
generuj_ciekawe_statystyki(MRegularSeasonDetailedResults, MNCAATourneyDetailedResults)

{'porównanie_ogólne': {'średnia_różnica_punktów_sezon': 11.990649522709516,
  'średnia_różnica_punktów_turniej': 11.678002894356005,
  'średnia_suma_punktów_sezon': 139.7672232224751,
  'średnia_suma_punktów_turniej': 139.18596237337192,
  'skuteczność_rzutów_zwycięzców_sezon': 47.48774807326924,
  'skuteczność_rzutów_zwycięzców_turniej': 47.55987912748867,
  'skuteczność_rzutów_przegranych_sezon': 40.276777473275764,
  'skuteczność_rzutów_przegranych_turniej': 39.72981141473538},
 'statystyki_roczne': [{'rok': 2003,
   'liczba_meczy_sezon': 4616,
   'liczba_meczy_turniej': 64,
   'średnia_różnica_punktów_sezon': 12.038128249566725,
   'średnia_różnica_punktów_turniej': 11.015625,
   'procent_meczy_z_dogrywką_sezon': 5.047660311958405,
   'procent_meczy_z_dogrywką_turniej': 7.8125},
  {'rok': 2004,
   'liczba_meczy_sezon': 4571,
   'liczba_meczy_turniej': 64,
   'średnia_różnica_punktów_sezon': 11.98096696565303,
   'średnia_różnica_punktów_turniej': 11.234375,
   'procent_meczy_z_dogr